#E-Commerce Data Preprocessing & Exploratory Data Analysis


### Dataset Description
The dataset is the **e-commerce clickstream dataset** from Kaggle: *"Ecommerce Behavior Data from Multi-Category Store"* (October & November 2019).
* **Source:** [Kaggle Link](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)
* **Scale:** Millions of rows representing user-product interactions.
* **Key Features:** `user_id`, `product_id`, `event_type`, `brand`, `price`, `category_code`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder
import os

warnings.simplefilter("ignore")

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ecommerce-behavior-data-from-multi-category-store' dataset.
Path to dataset files: /kaggle/input/ecommerce-behavior-data-from-multi-category-store


In [5]:

import os

for file in os.listdir(path):
    file_path = os.path.join(path, file)
    print(file, round(os.path.getsize(file_path) / (1024**2), 2), "MB")


2019-Nov.csv 8589.52 MB
2019-Oct.csv 5406.01 MB


In [8]:
oct_file = os.path.join(path, "2019-Oct.csv")
nov_file=os.path.join(path,"2019-Nov.csv")
df_oct = pd.read_csv(oct_file, nrows=10000)
df_nov=pd.read_csv(nov_file,nrows=10000)



In [11]:
for name, df in [("October", df_oct), ("November", df_nov)]:

    print(f"\n{'=' * 15} {name} {'=' * 15}")

    print("\n1. Dataset shape:")
    print(df.shape)

    print("\n2. Column names:")
    print(df.columns.tolist())

    print("\n3. Data types:")
    print(df.dtypes)

    print("\n8. First 5 rows:")
    display(df.head())


=============== October ===============

1. Dataset shape:
(10000, 9)

2. Column names:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

3. Data types:
event_time        object
event_type        object
product_id         int64
category_id        int64
category_code     object
brand             object
price            float64
user_id            int64
user_session      object
dtype: object

8. First 5 rows:


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d



=============== November ===============

1. Dataset shape:
(10000, 9)

2. Column names:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

3. Data types:
event_time        object
event_type        object
product_id         int64
category_id        int64
category_code     object
brand             object
price            float64
user_id            int64
user_session      object
dtype: object

8. First 5 rows:


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [12]:
def preprocess_chunk(df):
    # 1. Remove rows with missing values in required columns
    df = df.dropna(
        subset=[
            'user_id', 'product_id', 'event_type',
            'brand', 'price', 'category_code'
        ]
    )

    # 2. Remove duplicate user-product-event combinations
    df = df.drop_duplicates(
        subset=['user_id', 'product_id', 'event_type']
    )

    # 3. Keep only views and purchases
    df = df[df['event_type'].isin(['view', 'purchase'])]

    # 4. Treat IDs as identifiers, not numerical measurements
    df['user_id'] = df['user_id'].astype(str)
    df['product_id'] = df['product_id'].astype(str)

    # 5. Randomly retain 20% of the remaining rows
    df = df.sample(frac=0.2, random_state=42)

    # 6. Return the columns selected by the author
    return df[
        [
            'user_id', 'product_id', 'event_type',
            'brand', 'price', 'category_code'
        ]
    ]


In [ ]:
chunk_size=300_000
def process_file(file, output):
    for i, chunk in enumerate(pd.read_csv(file, chunksize=chunk_size)):
        processed_chunk = preprocess_chunk(chunk)
        processed_chunk.to_csv(
            output,
            mode='w' if i == 0 else 'a',
            header=(i == 0),
            index=False
        )

process_file(oct_file, 'processed_data1.csv')
process_file(nov_file, 'processed_data.csv')

In [15]:
# Load processed chunks
data_oct = pd.read_csv('processed_data.csv')
data_nov = pd.read_csv('processed_data1.csv')
full_data = pd.concat([data_oct, data_nov], ignore_index=True)

# Save the final Master Dataset for all models
full_data.to_csv("Ecommerce_Dataset.csv", index=False)

print(f"Final Dataset Shape: {full_data.shape}")
display(full_data.head())
display(full_data.describe())

Final Dataset Shape: (8366491, 6)


,user_id,product_id,event_type,brand,price,category_code
0,530394012,2701390,view,lg,1249.14,appliances.kitchen.refrigerators
1,563175289,1005122,view,apple,1027.59,electronics.smartphone
2,527560632,1003879,view,huawei,540.27,electronics.smartphone
3,556740096,4804660,view,xiaomi,23.05,electronics.audio.headphone
4,551479057,4804581,view,xiaomi,35.75,electronics.audio.headphone


,user_id,product_id,price
count,8.366491e+06,8.366491e+06,8.366491e+06
mean,5.373060e+08,7.138175e+06,3.652410e+02
std,2.159271e+07,1.303656e+07,3.967518e+02
min,2.951588e+07,1.000978e+06,8.700000e-01
25%,5.164106e+08,1.004958e+06,1.029400e+02
50%,5.341801e+08,1.801883e+06,2.185100e+02
75%,5.570696e+08,5.800108e+06,4.759100e+02
max,5.799698e+08,1.000284e+08,2.574070e+03
